### Step 5: Telling the Story with Data

Step 5 synthesises the earlier analysis into a coherent narrative. It uses one short evidence-check code cell and one synthesis graph, while keeping long explanations and insight cards in markdown so the full text remains visible in VS Code without truncation.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from textwrap import wrap

clean_df = pd.read_csv("../data/clean-survey.csv")

def pct(mask):
    return mask.mean() * 100

def wrap_labels(labels, width=18):
    return ["\n".join(wrap(str(label), width=width)) for label in labels]

frequency_scale = {"Rarely": 1, "Sometimes": 2, "Often": 3, "Always": 4}
stop_checking_num = clean_df["stop_checking_data_frequency"].map(frequency_scale)

print("Step 5 evidence check")
print("---------------------")
print(f"Cleaned sample size: n = {len(clean_df)}")
print(f"18-24 age group: {pct(clean_df['age_group'].eq('18-24')):.1f}%")
print(f"University students: {pct(clean_df['occupation'].eq('University student')):.1f}%")
print(f"Former or non-current users: {pct(clean_df['usage_duration'].eq('I do not currently use them')):.1f}%")
print(f"Understandable information high agreement: {pct(clean_df['understandable_information'].ge(3)):.1f}%")
print(f"Motivation high agreement: {pct(clean_df['motivated_feeling'].ge(3)):.1f}%")
print(f"Routine fit high agreement: {pct(clean_df['routine_fit'].ge(3)):.1f}%")
print(f"Actionable insight gap high agreement: {pct(clean_df['actionable_insight_gap'].ge(3)):.1f}%")
print(f"Stopped checking data sometimes or more: {pct(stop_checking_num.ge(2)):.1f}%")
print(f"Motivation and goal support Spearman correlation: {clean_df['motivated_feeling'].corr(clean_df['goal_consistency_support'], method='spearman'):.2f}")
print(f"Information overload and interpretation difficulty Spearman correlation: {clean_df['information_overload'].corr(clean_df['data_interpretation_difficulty'], method='spearman'):.2f}")
print(f"Information overload and stopping checking data Spearman correlation: {clean_df['information_overload'].corr(stop_checking_num, method='spearman'):.2f}")

#### 1. Inputs from Earlier Steps

| Earlier step | Relevant output for Step 5 | Evidence reused here |
|---|---|---|
| Step 1: Preparing the data | `clean-survey.csv` is the primary cleaned dataset. `dummy-data.csv` is a schema/reference file only. | `clean-survey.csv` contains `27` responses and `30` columns. The Likert-style variables used later are within the expected `0-4` range. |
| Step 2: Demographics | The sample is mostly young and student-based. | `88.9%` of respondents are aged `18-24`; `74.1%` are university students. |
| Step 3: Problem validation | Problems are moderate but recurring, especially around actionability, battery anxiety, visibility, and feedback usefulness. | Actionable insight gap and battery anxiety each have `44.4%` high agreement. Visibility difficulties, feedback not being useful, and stopping checking data are each reported sometimes or more by `77.8%`. |
| Step 4: Psychographics and behaviour | Participants often value and understand tracking data, but this does not guarantee routine fit or sustained engagement. | Understandable information and motivation each have `77.8%` high agreement. Routine fit has `33.3%` high agreement. Motivation and goal consistency support are associated (`Spearman = 0.59`). |

#### 2. Integrated Narrative Analysis

The respondent group is mostly young and student-based, so the findings are most useful for reasoning about student and young adult health-tracking contexts. Within this group, wearable use is mixed: just over half of respondents are former or non-current users, while the remaining respondents include both low-frequency and more regular users.

The main story is not that participants reject health-tracking data. Many participants report that the information is understandable and motivating. The more important issue is that this perceived value does not always translate into sustained, routine-compatible action.

In [ ]:
from matplotlib.patches import Patch

evidence_items = [
    ("18-24 age group", pct(clean_df["age_group"].eq("18-24")), "Sample context"),
    ("University students", pct(clean_df["occupation"].eq("University student")), "Sample context"),
    ("Non-current users", pct(clean_df["usage_duration"].eq("I do not currently use them")), "Usage context"),
    ("Understandable info", pct(clean_df["understandable_information"].ge(3)), "Perceived value"),
    ("Motivation", pct(clean_df["motivated_feeling"].ge(3)), "Perceived value"),
    ("Routine fit", pct(clean_df["routine_fit"].ge(3)), "Fit gap"),
    ("Actionability gap", pct(clean_df["actionable_insight_gap"].ge(3)), "Action gap"),
    ("Stopped checking", pct(stop_checking_num.ge(2)), "Engagement gap"),
]

color_map = {
    "Sample context": "#7fb3d5",
    "Usage context": "#3d7ea6",
    "Perceived value": "#5c946e",
    "Fit gap": "#d1495b",
    "Action gap": "#d1495b",
    "Engagement gap": "#d1495b",
}

evidence_items = sorted(evidence_items, key=lambda item: item[1])
labels = [label for label, _, _ in evidence_items]
values = [value for _, value, _ in evidence_items]
colors = [color_map[group] for _, _, group in evidence_items]

fig, ax = plt.subplots(figsize=(12, 7), constrained_layout=True)
bars = ax.barh(range(len(values)), values, color=colors)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(wrap_labels(labels, width=18))
ax.set_xlim(0, 100)
ax.set_xlabel("Participants (%)")
ax.set_title("Synthesis Evidence Map")
ax.bar_label(bars, labels=[f"{value:.1f}%" for value in values], padding=4)
ax.legend(
    handles=[
        Patch(facecolor="#7fb3d5", label="Sample context"),
        Patch(facecolor="#3d7ea6", label="Usage context"),
        Patch(facecolor="#5c946e", label="Perceived value"),
        Patch(facecolor="#d1495b", label="Gap / friction"),
    ],
    loc="lower right",
    frameon=False,
)
plt.show()

#### 3. Key Insights

1. **Users value health-tracking data, but value does not guarantee action.** Understandable information and motivation are high, while the actionable insight gap remains visible.

2. **Ongoing engagement is a problem, not only initial adoption.** Stopping checking data is common, including among current users.

3. **Routine fit is weaker than motivation.** Users may feel motivated by progress data while still finding tracking difficult to integrate into daily life.

4. **Information load appears connected to disengagement.** Step 4 found that information overload is associated with interpretation difficulty and stopping checking data.

5. **Goal-linked feedback is a promising direction.** Motivation is associated with goal consistency support, suggesting that feedback may be more useful when connected to personally meaningful goals.

#### 4. Insight Card Drafts

##### Insight Card 1: Value Without Action Is Not Enough

**Short explanation:** Participants often understand and value tracking information, but this does not always translate into clear next steps.

**Supporting evidence:** Understandable information and motivation each have `77.8%` high agreement. Actionable insight gap has `44.4%` high agreement.

**Why it matters:** A system can be informative but still fail to support behaviour change if users are left to interpret what to do next.

**Design implication:** Prioritise feedback that translates data into small, context-appropriate actions rather than adding more raw metrics.

##### Insight Card 2: Engagement Can Weaken During Continued Use

**Short explanation:** Stopping checking data is not limited to people who no longer use wearables.

**Supporting evidence:** Current users average `2.54` for stopping checking data, compared with `2.00` for former or non-current users. Across all participants, `77.8%` stop checking sometimes or more.

**Why it matters:** Retention should be treated as an ongoing experience problem, not only a problem after users abandon the technology.

**Design implication:** Support lightweight re-engagement and make important feedback visible without requiring repeated manual checking.

##### Insight Card 3: Motivation Needs Routine Fit

**Short explanation:** Users may be motivated by tracking but still find the technology difficult to integrate into everyday routines.

**Supporting evidence:** Motivation has `77.8%` high agreement, while routine fit has `33.3%` high agreement.

**Why it matters:** Interest alone may not sustain use when checking, charging, interpreting, or acting on data feels inconvenient.

**Design implication:** Design for low-burden interactions that fit around existing activities rather than adding extra tasks.

##### Insight Card 4: Overload Can Undermine Interpretation

**Short explanation:** Information overload appears connected to interpretation difficulty and disengagement from checking behaviour.

**Supporting evidence:** Information overload is associated with interpretation difficulty (`Spearman = 0.47`) and stopping checking data (`Spearman = 0.37`).

**Why it matters:** More information may not improve the experience if it increases cognitive effort or makes priorities unclear.

**Design implication:** Prioritise summarisation, hierarchy, and timely recommendations over dense dashboards.

##### Insight Card 5: Goal Support Is a Motivation Pathway

**Short explanation:** Motivation appears stronger when participants feel technology supports consistent goal pursuit.

**Supporting evidence:** Motivation and goal consistency support have a Spearman correlation of `0.59`; goal support has `48.1%` high agreement.

**Why it matters:** Progress data is more meaningful when users can connect it to goals they are trying to maintain.

**Design implication:** Frame tracking around achievable personal goals, progress interpretation, and next-step support.

#### 5. Design Implications

The later wearable or mobile design should support actionable feedback, routine-compatible engagement, goal-linked motivation, information prioritisation, and lightweight re-engagement.

The design should avoid adding more raw data without interpretation, treating notifications as a complete solution by themselves, assuming current use equals sustained value, or overgeneralising beyond a mostly young and student-based sample.

#### Step 5 Reflection

The overall story from Steps 1-5 is that participants do not simply reject wearable or health-tracking technology. Many can understand the information and feel motivated by it. The unmet need is more specific: users need health data that is easier to interpret, easier to act on, and easier to sustain within everyday routines.